# CUB final locked evaluation
Run from the top in a GPU Kaggle session. Every subprocess writes complete stdout/stderr logs, failures include the real stderr tail, and official-test model work cannot begin until both exact-revision training-only smokes validate and the protocol locks. Long shared runs use atomic shards and are safe to resume by running the notebook again.

In [ ]:
import json, os, platform, subprocess, sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)
import torch
TORCH_BEFORE = torch.__version__
print({'torch': TORCH_BEFORE, 'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count(), 'cudnn': torch.backends.cudnn.version(), 'bf16': torch.cuda.is_available() and torch.cuda.is_bf16_supported()})
if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle CUDA accelerator is required before any model work')
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    free_bytes, total_bytes = torch.cuda.mem_get_info(index)
    print({'index': index, 'name': properties.name, 'capability': torch.cuda.get_device_capability(index), 'property_total_GiB': round(properties.total_memory / 2**30, 3), 'free_GiB': round(free_bytes / 2**30, 3), 'total_GiB': round(total_bytes / 2**30, 3)})
for distribution in ('transformers', 'accelerate', 'bitsandbytes', 'huggingface-hub', 'safetensors'):
    try:
        print('pre-install', distribution, package_version(distribution))
    except PackageNotFoundError:
        print('pre-install', distribution, 'NOT INSTALLED')

In [ ]:
REPO_URL = 'https://github.com/Ram21275/newpipeline.git'
BRANCH = 'feat/iclr'
REQUIRED_BASE_COMMIT = '88485e926a25f2999567fdf767274ab2de07f4df'
REPO_ROOT = Path('/kaggle/working/newpipeline')
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
ancestor = subprocess.run(['git', '-C', str(REPO_ROOT), 'merge-base', '--is-ancestor', REQUIRED_BASE_COMMIT, COMMIT])
if ancestor.returncode != 0:
    raise RuntimeError(f'{COMMIT} is not descended from required base {REQUIRED_BASE_COMMIT}')
PROJECT = REPO_ROOT / 'projects/logit_evidence_routing'
if not (PROJECT / 'pyproject.toml').is_file():
    raise RuntimeError(f'Project checkout is incomplete: {PROJECT}')
WORK = Path('/kaggle/working/cub_final')
for name in ('environment/logs', 'manifests', 'smoke', 'results', 'analysis', 'archive'):
    (WORK / name).mkdir(parents=True, exist_ok=True)
(WORK / 'environment/source_commit.txt').write_text(COMMIT + '\n', encoding='ascii')
print({'PROJECT': str(PROJECT), 'COMMIT': COMMIT, 'WORK': str(WORK)})

## Install and immutable environment checks
Both installs use `--no-deps`; they cannot replace Kaggle's PyTorch/CUDA stack. The exact experiment-package versions are then checked in a fresh subprocess.

In [ ]:
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
LOG_DIR = WORK / 'environment/logs'

def run_stage(stage, command):
    print('RUN', stage, command)
    completed = subprocess.run(command, text=True, capture_output=True)
    stdout_path = LOG_DIR / f'{stage}.stdout.txt'
    stderr_path = LOG_DIR / f'{stage}.stderr.txt'
    stdout_path.write_text(completed.stdout, encoding='utf-8')
    stderr_path.write_text(completed.stderr, encoding='utf-8')
    print({'stage': stage, 'returncode': completed.returncode, 'stdout_log': str(stdout_path), 'stderr_log': str(stderr_path)})
    if completed.stdout:
        print('STDOUT tail:\n' + completed.stdout[-6000:])
    if completed.stderr and completed.returncode == 0:
        print('STDERR tail:\n' + completed.stderr[-6000:])
    if completed.returncode != 0:
        raise RuntimeError(f"Stage {stage!r} failed with exit code {completed.returncode}.\nCommand: {command!r}\nComplete logs: {stdout_path} and {stderr_path}\n----- STDOUT TAIL -----\n{completed.stdout[-12000:]}\n----- STDERR TAIL -----\n{completed.stderr[-30000:]}")
    return completed

run_stage('install_editable', [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '-e', str(PROJECT)])
run_stage('install_pinned_packages', [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '-r', str(PROJECT / 'requirements-cub-final-kaggle.txt')])
import torch as torch_after
if torch_after.__version__ != TORCH_BEFORE:
    raise RuntimeError(f'PyTorch changed unexpectedly: {TORCH_BEFORE} -> {torch_after.__version__}')
required_versions = {'transformers': '4.57.1', 'accelerate': '1.11.0', 'bitsandbytes': '0.48.1', 'huggingface-hub': '0.36.2', 'safetensors': '0.6.2'}
runtime_probe_code = '''
import accelerate
import bitsandbytes
import huggingface_hub
import json
import safetensors
import transformers
from huggingface_hub.constants import HF_HUB_CACHE
observed = {
    'transformers': transformers.__version__,
    'accelerate': accelerate.__version__,
    'bitsandbytes': bitsandbytes.__version__,
    'huggingface-hub': huggingface_hub.__version__,
    'safetensors': safetensors.__version__,
}
print('CUB_FINAL_VERSION_JSON=' + json.dumps({'versions': observed, 'hf_hub_cache': HF_HUB_CACHE}, sort_keys=True))
'''
runtime_probe = run_stage('runtime_import_probe', [sys.executable, '-c', runtime_probe_code])
version_lines = [line for line in runtime_probe.stdout.splitlines() if line.startswith('CUB_FINAL_VERSION_JSON=')]
if len(version_lines) != 1:
    raise RuntimeError(f'Runtime import probe did not emit exactly one result: {runtime_probe.stdout!r}')
version_probe_payload = json.loads(version_lines[0].split('=', 1)[1])
observed_versions = version_probe_payload['versions']
version_problems = {
    name: observed
    for name, observed in observed_versions.items()
    if observed != required_versions[name]
}
print('FRESH-PROCESS PACKAGE VERSIONS', observed_versions)
if version_problems:
    raise RuntimeError(f'Runtime package mismatch: expected {required_versions}, problems {version_problems}')
run_stage('package_origin', [sys.executable, '-c', 'import cub_final, cub_final.scoring; print(cub_final.__file__); print(cub_final.scoring.__file__)'])
run_stage('inventory', [sys.executable, '-m', 'cub_final', 'inventory', '--output', str(WORK / 'environment/inventory.json')])
HF_HUB_CACHE = version_probe_payload['hf_hub_cache']
model_revisions = {
    'llava-hf/llava-1.5-7b-hf': 'b234b804b114d9e37bb655e11cbbb5f5e971b7a9',
    'Qwen/Qwen3-VL-2B-Instruct': '89644892e4d85e24eaac8bacfd4f463576704203',
}
for checkpoint, revision in model_revisions.items():
    snapshot = Path(HF_HUB_CACHE) / ('models--' + checkpoint.replace('/', '--')) / 'snapshots' / revision
    print({'checkpoint': checkpoint, 'revision': revision, 'exact_snapshot_cached': snapshot.is_dir(), 'snapshot_path': str(snapshot), 'cached_entries': sum(1 for _ in snapshot.rglob('*')) if snapshot.is_dir() else 0})

## Input integrity, manifests, and smoke validators
The audit rejects an images-only mirror and verifies the narrowly normalized 606 official six-column anomaly rows. Manifest construction reads annotations only; no official-test image is sent to a model before the lock.

In [ ]:
required_cub_files = [
    'images.txt', 'image_class_labels.txt', 'train_test_split.txt', 'classes.txt',
    'bounding_boxes.txt', 'parts/parts.txt', 'parts/part_locs.txt',
    'attributes/attributes.txt', 'attributes/certainties.txt',
    'attributes/image_attribute_labels.txt',
]
preferred = Path('/kaggle/input/datasets/wenewone/cub2002011/CUB_200_2011')
candidate_roots = [preferred] if preferred.is_dir() else sorted({p.parent for p in Path('/kaggle/input').rglob('train_test_split.txt')})
valid_roots = [root for root in candidate_roots if (root / 'images').is_dir() and all((root / name).is_file() for name in required_cub_files)]
for root in candidate_roots:
    print({'candidate': str(root), 'images': (root / 'images').is_dir(), 'missing': [name for name in required_cub_files if not (root / name).is_file()]})
if len(valid_roots) != 1:
    raise RuntimeError(f'Expected exactly one complete CUB root, found {valid_roots}')
CUB_ROOT = valid_roots[0]
run_stage('cub_audit', [sys.executable, '-m', 'cub_final', 'audit', '--cub-root', str(CUB_ROOT), '--output', str(WORK / 'manifests/cub_integrity_audit.json')])
run_stage('build_manifests', [sys.executable, '-m', 'cub_final', 'manifests', '--cub-root', str(CUB_ROOT), '--output-dir', str(WORK / 'manifests')])
audit = json.loads((WORK / 'manifests/cub_integrity_audit.json').read_text(encoding='utf-8'))
if audit.get('status') != 'PASS' or audit.get('known_six_column_attribute_rows_normalized') != 606:
    raise RuntimeError(f'CUB audit did not validate the locked dataset: {audit}')
manifest_summary = json.loads((WORK / 'manifests/manifest_summary.json').read_text(encoding='utf-8'))
print('CUB_ROOT=', CUB_ROOT)
print(json.dumps(manifest_summary, indent=2, sort_keys=True))

SMOKE_SPECS = {
    'llava': {'checkpoint': 'llava-hf/llava-1.5-7b-hf', 'revision': 'b234b804b114d9e37bb655e11cbbb5f5e971b7a9', 'projection': 'lm_head'},
    'qwen3': {'checkpoint': 'Qwen/Qwen3-VL-2B-Instruct', 'revision': '89644892e4d85e24eaac8bacfd4f463576704203', 'projection': 'final_norm_then_lm_head'},
}

def validate_smoke(path, architecture):
    if not path.is_file():
        raise RuntimeError(f'Missing smoke output: {path}')
    report = json.loads(path.read_text(encoding='utf-8'))
    spec = SMOKE_SPECS[architecture]
    audit_value = report.get('architecture_audit') or {}
    runtime = audit_value.get('runtime') or {}
    decision = report.get('training_decision') or {}
    measurement = report.get('measurement') or {}
    checks = {
        'status_pass': report.get('status') == 'PASS',
        'official_test_images_used_zero': report.get('official_test_images_used') == 0,
        'checkpoint_matches': report.get('checkpoint') == spec['checkpoint'],
        'required_revision_matches': report.get('required_revision') == spec['revision'],
        'unquantized_request': report.get('requested_quantization') == 'none',
        'architecture_matches': audit_value.get('architecture') == architecture == measurement.get('architecture'),
        'resolved_revision_matches': audit_value.get('resolved_revision') == spec['revision'] == measurement.get('resolved_revision'),
        'runtime_unquantized': runtime.get('quantization') == 'none',
        'training_split_only': decision.get('official_split') == 'train',
        'final_projection_matches': measurement.get('final_hidden_projection') == spec['projection'],
        'final_logit_agreement_present': isinstance(measurement.get('final_logit_agreement'), dict) and 'max_abs_error' in measurement.get('final_logit_agreement', {}),
    }
    if not all(checks.values()):
        raise RuntimeError(f'{architecture} smoke validation failed: {checks}')
    return report

def ensure_smoke(architecture):
    path = WORK / f'smoke/{architecture}_smoke.json'
    try:
        report = validate_smoke(path, architecture)
        print(f'Reusing validated {architecture} training-only smoke: {path}')
        return report
    except Exception as prior:
        print(f'{architecture} smoke must run: {prior}')
    run_stage(f'{architecture}_smoke', [sys.executable, '-m', 'cub_final', 'smoke', '--architecture', architecture, '--cub-root', str(CUB_ROOT), '--output', str(path)])
    report = validate_smoke(path, architecture)
    print(f'VALIDATED {architecture}: training image {report["training_decision"]["image_id"]}; official-test images used = 0')
    return report

## Training-only real-model smoke checks
Models load sequentially with unquantized bf16/fp16 inference. There is no automatic quantization fallback because that would change the locked measurement scope.

In [ ]:
llava_smoke = ensure_smoke('llava')

In [ ]:
qwen3_smoke = ensure_smoke('qwen3')

In [ ]:
decisions = int(manifest_summary['expensive_decision_count'])
for architecture, report, arms in (('llava', llava_smoke, 6), ('qwen3', qwen3_smoke, 7)):
    measurement = report['measurement']
    print({'architecture': architecture, 'instrumented_forward_seconds': round(measurement['runtime_seconds'], 3), 'visual_tokens': measurement['visual_token_count'], 'max_gpu_memory_GiB': round(measurement['max_gpu_memory_bytes'] / 2**30, 3), 'rough_unbatched_arm_hours': round(measurement['runtime_seconds'] * decisions * arms / 3600, 1)})

## Automatic protocol lock
This fails closed unless both smoke files prove official-training-only use, exact checkpoints/revisions, unquantized runtime, and final-logit reconstruction agreement.

In [ ]:
validate_smoke(WORK / 'smoke/llava_smoke.json', 'llava')
validate_smoke(WORK / 'smoke/qwen3_smoke.json', 'qwen3')
run_stage('lock_protocol', [sys.executable, '-m', 'cub_final', 'lock', '--manifest-dir', str(WORK / 'manifests'), '--smoke-dir', str(WORK / 'smoke'), '--output', str(WORK / 'FINAL_PROTOCOL.yaml')])
protocol = json.loads((WORK / 'FINAL_PROTOCOL.yaml').read_text(encoding='utf-8'))
if protocol.get('status') != 'LOCKED':
    raise RuntimeError('FINAL_PROTOCOL.yaml did not lock')
print('protocol_hash=', protocol['protocol_hash'])

## Locked shared experiment: LLaVA then Qwen3
The following cells are the first model evaluations that may open official-test images. Each condition is an atomic shard. Rerunning retries failed required conditions, preserves completed current-code shards, and permits only explicit missing-part oracle exclusions.

In [ ]:
questions = WORK / 'manifests/question_manifest_expensive.jsonl'
question_count = sum(1 for line in questions.read_text(encoding='utf-8').splitlines() if line.strip())

def validate_shared(architecture):
    path = WORK / f'results/{architecture}/shared_per_example_shards.jsonl'
    if not path.is_file():
        raise RuntimeError(f'Missing consolidated shared output: {path}')
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    expected = question_count * (7 if architecture == 'qwen3' else 6)
    keys = [row.get('key') for row in rows]
    failed = [key for key, row in zip(keys, rows) if row.get('status') == 'failed']
    bad_exclusions = [key for key, row in zip(keys, rows) if row.get('status') == 'excluded' and not str(key).endswith('::oracle_part_crop')]
    checks = {'row_count': len(rows) == expected, 'unique_keys': len(set(keys)) == expected, 'no_failed': not failed, 'only_oracle_excluded': not bad_exclusions, 'one_config': len({row.get('config_hash') for row in rows}) == 1}
    if not all(checks.values()):
        raise RuntimeError(f'{architecture} shared output failed validation: checks={checks}, failed={failed[:5]}, bad_exclusions={bad_exclusions[:5]}')
    print({'architecture': architecture, 'validated_terminal_shards': len(rows), 'complete': sum(row.get('status') == 'complete' for row in rows), 'excluded_oracle': sum(row.get('status') == 'excluded' for row in rows), 'path': str(path)})
    return path

run_stage('llava_shared', [sys.executable, '-m', 'cub_final', 'run-shared', '--architecture', 'llava', '--cub-root', str(CUB_ROOT), '--protocol', str(WORK / 'FINAL_PROTOCOL.yaml'), '--questions', str(questions), '--output-dir', str(WORK / 'results')])
llava_output = validate_shared('llava')

In [ ]:
run_stage('qwen3_shared', [sys.executable, '-m', 'cub_final', 'run-shared', '--architecture', 'qwen3', '--cub-root', str(CUB_ROOT), '--protocol', str(WORK / 'FINAL_PROTOCOL.yaml'), '--questions', str(questions), '--output-dir', str(WORK / 'results')])
qwen3_output = validate_shared('qwen3')

## Consolidate, cluster-bootstrap, figures, and archive
Analysis is unreachable unless both complete consolidated model outputs pass the coverage gate above. DoLa records retain ordinary generation and label contrastive scores as non-probabilities.

In [ ]:
llava_output = validate_shared('llava')
qwen3_output = validate_shared('qwen3')
combined = WORK / 'results/all_shared_per_example.jsonl'
temporary = combined.with_suffix('.jsonl.tmp')
with temporary.open('w', encoding='utf-8') as output_handle:
    for path in (llava_output, qwen3_output):
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                output_handle.write(line + '\n')
temporary.replace(combined)
run_stage('analysis', [sys.executable, '-m', 'cub_final', 'analyze', '--input', str(combined), '--output-dir', str(WORK / 'analysis'), '--bootstrap-resamples', '10000'])
analysis = json.loads((WORK / 'analysis/final_analysis.json').read_text(encoding='utf-8'))
if analysis.get('status') != 'COMPLETE_FROM_PROVIDED_PER_EXAMPLE_OUTPUTS' or analysis.get('models') != ['llava', 'qwen3']:
    raise RuntimeError(f'Analysis validation failed: {analysis}')
run_stage('figures', [sys.executable, '-m', 'cub_final', 'figures', '--analysis', str(WORK / 'analysis/final_analysis.json'), '--output-dir', str(WORK / 'analysis/figures')])
print(json.dumps(analysis, indent=2, sort_keys=True)[:6000])

In [ ]:
archive = WORK / 'archive/cub_final_results.tar.gz'
run_stage('archive', [sys.executable, '-m', 'cub_final', 'archive', '--root', str(WORK), '--include', 'environment', '--include', 'manifests', '--include', 'smoke', '--include', 'FINAL_PROTOCOL.yaml', '--include', 'results', '--include', 'analysis', '--output', str(archive)])
archive_audit_path = Path(str(archive) + '.audit.json')
archive_audit = json.loads(archive_audit_path.read_text(encoding='utf-8'))
if archive_audit.get('status') != 'PASS' or not archive.is_file() or not Path(str(archive) + '.sha256').is_file():
    raise RuntimeError(f'Archive validation failed: {archive_audit}')
print('VALIDATED ARCHIVE — save notebook output now:', archive, Path(str(archive) + '.sha256'), archive_audit_path)

## Remaining final-paper stages
This notebook completes only the locked shared re-encoding/decoding track. Frozen-feature probes and model-internal selective interventions still require their architecture-specific Qwen3 hooks and audited final-test outputs; this notebook does not represent them as completed.